In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('ADEME_dpe-v2-tertiaire-2.csv')

In [3]:
score = 0

for col in df.columns:
    if df[col].isna().sum() > 4999:
        score += 1

print(score)

22


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 63 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   N°DPE                              10000 non-null  str    
 1   Date_réception_DPE                 10000 non-null  str    
 2   Date_établissement_DPE             10000 non-null  str    
 3   Date_visite_diagnostiqueur         10000 non-null  str    
 4   Modèle_DPE                         10000 non-null  str    
 5   N°_DPE_remplacé                    582 non-null    str    
 6   Date_fin_validité_DPE              10000 non-null  str    
 7   Version_DPE                        10000 non-null  float64
 8   N°_DPE_immeuble_associé            26 non-null     str    
 9   Méthode_du_DPE                     9585 non-null   str    
 10  N°_immatriculation_copropriété     34 non-null     str    
 11  Invariant_fiscal_logement          60 non-null     str    
 12  Et

In [9]:
df[['Surface_utile', 'Version_DPE', 'Nombre_occupant']].describe()

,Surface_utile,Version_DPE,Nombre_occupant
count,10000.000000,10000.000000,9.455000e+03
mean,423.974620,1.983680,3.122633e+04
std,2524.285877,0.402263,3.011636e+06
min,2.400000,1.000000,0.000000e+00
25%,100.000000,2.100000,0.000000e+00
50%,100.000000,2.100000,1.000000e+00
75%,100.000000,2.200000,6.850000e+01
max,116912.000000,2.300000,2.928417e+08


In [10]:
col = 'Emission_GES_kgCO2/m²/an'
df = df[~df[col].isna()].copy()
df.reset_index(inplace = True, drop = True)

In [11]:
from scipy import stats
df['z_emissions'] = stats.zscore(df[col])


In [13]:
df.loc[df['z_emissions'] > 3, 'z_emissions'].count()

np.int64(11)

In [14]:
df.loc[df['z_emissions'] > 2, 'z_emissions'].count()

np.int64(37)

In [15]:
col = 'Emission_GES_kgCO2/m²/an'
iqr = np.quantile(df[col], q=[0.25, 0.75])
limb = iqr[0] - 1.5*(iqr[1] - iqr[0])
limh = iqr[1] + 1.5*(iqr[1] - iqr[0])

In [16]:
print(limb)
print(limh)

-20.950000000000003
44.25


In [17]:
df.loc[df[col] > limh, col].count()

np.int64(488)

In [18]:
pd.cut(df[col], 10).value_counts()

Emission_GES_kgCO2/m²/an
(-3.126, 312.59]      5604
(312.59, 625.18]         1
(1250.36, 1562.95]       1
(1562.95, 1875.54]       1
(2813.31, 3125.9]        1
(625.18, 937.77]         0
(937.77, 1250.36]        0
(1875.54, 2188.13]       0
(2188.13, 2500.72]       0
(2500.72, 2813.31]       0
Name: count, dtype: int64

In [19]:
pd.qcut(df[col], 10).value_counts()

Emission_GES_kgCO2/m²/an
(-0.001, 0.1]      642
(4.3, 6.0]         574
(6.0, 8.0]         567
(40.43, 3125.9]    561
(2.7, 4.3]         560
(10.7, 15.8]       560
(15.8, 24.5]       560
(24.5, 40.43]      558
(8.0, 10.7]        528
(0.1, 2.7]         498
Name: count, dtype: int64

In [21]:
df.Secteur_activité.unique()

<StringArray>
[                                                                                       'U : Établissements de soins',
                                                                              'W : Administrations, banques, bureaux',
                                                                     'T : Salles d'exposition à vocation commerciale',
                                                                                          'autres tertiaires non ERP',
                                                                               'N : Restaurants et débits de boisson',
                                                                                                      'GHW : Bureaux',
                                                                         'M : Magasins de vente, centres commerciaux',
                                                                                      'locaux d'entreprise (bureaux)',
                                  